In [1]:
import os
import pandas as pd

# Load the cleaned dataset saved by part-1_part-2
file_path = 'Final tables/cleaned-data.csv'
data = pd.read_csv(file_path)
print(data.shape)

(43193, 15)


In [2]:
# Is there a natural primary key? Check whether every row is unique.
duplicate_rows = data.duplicated().sum()
if duplicate_rows == 0:
    print("Each row is unique, so the whole record could act as a primary key if no simpler key is found.")
else:
    print(f"{duplicate_rows} row is an exact copy of another, so no combination of columns can identify a call.")

1 row is an exact copy of another, so no combination of columns can identify a call.


In [3]:
# Nothing in the data identifies a call, so each call gets an id of its own
data.insert(0, 'id', range(1, len(data) + 1))
data.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,month,duration,pdays,previous,poutcome,y
0,1,58,management,married,tertiary,no,2143,yes,no,telephone,may,261,-1,0,other,0
1,2,44,technician,single,secondary,no,29,yes,no,telephone,may,151,-1,0,other,0
2,3,33,entrepreneur,married,secondary,no,2,yes,yes,telephone,may,76,-1,0,other,0
3,4,35,management,married,tertiary,no,231,yes,no,telephone,may,139,-1,0,other,0
4,5,28,management,single,tertiary,no,447,yes,yes,telephone,may,217,-1,0,other,0


### Is `poutcome` decided by `pdays` and `previous`?

That was the first idea for third normal form: if the result of the previous campaign
followed from the other two previous-campaign columns, it would be a transitive
dependency, and the three columns could move to a small lookup table.

In [4]:
# Count the outcomes that appear for each (pdays, previous) pair
outcomes_per_pair = data.groupby(['pdays', 'previous'])['poutcome'].nunique()

print("Different (pdays, previous) pairs:", len(outcomes_per_pair))
print("Pairs that appear with more than one poutcome:", (outcomes_per_pair > 1).sum())

# One example
example = outcomes_per_pair[outcomes_per_pair > 1].index[0]
print(f"Example: pdays={example[0]}, previous={example[1]} ->",
      sorted(data[(data['pdays'] == example[0]) & (data['previous'] == example[1])]['poutcome'].unique()))

Different (pdays, previous) pairs: 2404
Pairs that appear with more than one poutcome: 903
Example: pdays=1, previous=1 -> ['failure', 'other', 'success']


It is not. The same pair of values turns up with different outcomes, so there is no
dependency to remove, and a lookup table keyed on those two columns would be wrong.
With `id` as the key, nothing was found that breaks third normal form.

### What is repeated

Most people had never been contacted before. For them the file fills the three
previous-campaign columns with placeholders.

In [5]:
never_contacted = data['pdays'] == -1

print(f"Calls with no previous campaign (pdays = -1): {never_contacted.sum()} ({never_contacted.mean():.1%})")
print("...of those, previous = 0:", (data.loc[never_contacted, 'previous'] == 0).sum())
print("...and their poutcome:", data.loc[never_contacted, 'poutcome'].unique())
print("Calls with previous = 0 but a real pdays:", ((data['previous'] == 0) & ~never_contacted).sum())

Calls with no previous campaign (pdays = -1): 35281 (81.7%)
...of those, previous = 0: 35281
...and their poutcome: ['other']
Calls with previous = 0 but a real pdays: 0


So on four calls in five, three columns say one thing three times: there was no previous
campaign. The previous-campaign columns move to their own table, which has a row only
when there is something to record. A call with no row there has no history.

In [6]:
# Main table: one row per call
main_table = data.drop(columns=['pdays', 'previous', 'poutcome'])

# Previous-outcome table: only the calls that have a previous campaign, linked by the call id
p_outcome_table = data.loc[~never_contacted, ['id', 'pdays', 'previous', 'poutcome']]

print("Main table:", main_table.shape)
print("Previous-outcome table:", p_outcome_table.shape)

Main table: (43193, 13)
Previous-outcome table: (7912, 4)


In [7]:
# Check that nothing was lost: joining the tables back on id, and filling the gaps
# with the same placeholders, has to give the table we started with
rebuilt = main_table.merge(p_outcome_table, on='id', how='left')
rebuilt['pdays'] = rebuilt['pdays'].fillna(-1).astype('int64')
rebuilt['previous'] = rebuilt['previous'].fillna(0).astype('int64')
rebuilt['poutcome'] = rebuilt['poutcome'].fillna('other')

print("Same as before the split:", rebuilt[data.columns].equals(data))

Same as before the split: True


In [8]:
# Save the normalized tables
os.makedirs('Final tables', exist_ok=True)
main_table.to_csv('Final tables/main-table.csv', index=False)
p_outcome_table.to_csv('Final tables/p-outcome-table.csv', index=False)

print("Normalization steps completed. Main table and previous-outcome table have been saved separately.")

Normalization steps completed. Main table and previous-outcome table have been saved separately.